### 1. Data download

This notebook download the data using the NASA API. 

The NASA Exoplanets Archive supports programmatic TAP access, and the Kepler Objects of Interest (KOI) cumulative table is available as `cumulative`; TAP queries can return CSV, JSON,  TSV or VOTable output. The tagret column we'll use is `koi_dissipation`, whose official values include `CANDIDATE`, `FALSE POSITIVE`, `NOT DISPOSITIONED` and `CONFIRMED`.

In [24]:
import io
import os
import json
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

In [25]:
# define project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# define project data directory
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
REPORTS_DIR = DATA_DIR / 'reports'

# create dirs
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# prints paths
print('Project root:', PROJECT_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('REPORTS_DIR:', REPORTS_DIR)


Project root: d:\exoplanet-ml-api
RAW_DIR: d:\exoplanet-ml-api\data\raw
PROCESSED_DIR: d:\exoplanet-ml-api\data\processed
REPORTS_DIR: d:\exoplanet-ml-api\data\reports


## Define NASA TAP query
We’ll download a controlled set of columns rather than the entire table.

In [26]:
TAP_SYNC_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

SELECT_COLUMNS = [
    # Identifiers
    "kepid",
    "kepoi_name",
    "kepler_name",

    # Target / label
    "koi_disposition",

    # Transit/orbital properties
    "koi_period",
    "koi_impact",
    "koi_duration",
    "koi_depth",
    "koi_prad",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_count",
    "koi_num_transits",

    # Stellar properties
    "koi_steff",
    "koi_slogg",
    "koi_smet",
    "koi_srad",
    "koi_smass",

    # Sky position and magnitude
    # These are useful for EDA, but we may choose not to use them in the first model.
    "ra",
    "dec",
    "koi_kepmag",
]

query = f"""
SELECT {", ".join(SELECT_COLUMNS)}
FROM cumulative
"""

print(query)


SELECT kepid, kepoi_name, kepler_name, koi_disposition, koi_period, koi_impact, koi_duration, koi_depth, koi_prad, koi_teq, koi_insol, koi_model_snr, koi_count, koi_num_transits, koi_steff, koi_slogg, koi_smet, koi_srad, koi_smass, ra, dec, koi_kepmag
FROM cumulative



In [27]:
def download_tap_csv(query: str, tap_url: str = TAP_SYNC_URL) -> pd.DataFrame:
    """
    Download the result of a TAP query as a CSV file.

    Parameters:
    - query: The TAP query to execute.
    - tap_url: The URL of the TAP service.

    Returns:
    - The path to the downloaded CSV file.
    """
    # Define the parameters for the TAP request
    params = {
        "query": " ".join(query.split()),
        "format": "csv",
    }
    response = requests.get(tap_url, params=params, timeout=120)
    response.raise_for_status()

    # basic check for an HTML/XML error page
    text_start = response.text[:100].lower()
    if "<html" in text_start or "<error" in text_start:
        raise RuntimeError("TAP returned an HTML/XML error page instead of CSV data.\n")
    return pd.read_csv(io.StringIO(response.text))

df_raw = download_tap_csv(query)
print("Shape:", df_raw.shape)
df_raw.head()


Shape: (9564, 22)


,kepid,kepoi_name,kepler_name,koi_disposition,koi_period,koi_impact,koi_duration,koi_depth,koi_prad,koi_teq,...,koi_count,koi_num_transits,koi_steff,koi_slogg,koi_smet,koi_srad,koi_smass,ra,dec,koi_kepmag
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,9.488036,0.146,2.95750,615.8,2.26,793.0,...,2,142.0,5455.0,4.467,0.14,0.927,0.919,291.93423,48.141651,15.347
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,54.418383,0.586,4.50700,874.8,2.83,443.0,...,2,25.0,5455.0,4.467,0.14,0.927,0.919,291.93423,48.141651,15.347
2,10811496,K00753.01,NaN,CANDIDATE,19.899140,0.969,1.78220,10829.0,14.60,638.0,...,1,56.0,5853.0,4.544,-0.18,0.868,0.961,297.00482,48.134129,15.436
3,10848459,K00754.01,NaN,FALSE POSITIVE,1.736952,1.276,2.40641,8079.2,33.46,1395.0,...,1,621.0,5805.0,4.564,-0.52,0.791,0.836,285.53461,48.285210,15.597
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,2.525592,0.701,1.65450,603.3,2.75,1406.0,...,1,515.0,6031.0,4.438,0.07,1.046,1.095,288.75488,48.226200,15.509


In [28]:
# save the raw data
df_raw.to_csv(RAW_DIR / 'cumulative.csv', index=False)

In [29]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9564 entries, 0 to 9563
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   kepid             9564 non-null   int64  
 1   kepoi_name        9564 non-null   object 
 2   kepler_name       2747 non-null   object 
 3   koi_disposition   9564 non-null   object 
 4   koi_period        9564 non-null   float64
 5   koi_impact        9201 non-null   float64
 6   koi_duration      9564 non-null   float64
 7   koi_depth         9201 non-null   float64
 8   koi_prad          9201 non-null   float64
 9   koi_teq           9201 non-null   float64
 10  koi_insol         9243 non-null   float64
 11  koi_model_snr     9201 non-null   float64
 12  koi_count         9564 non-null   int64  
 13  koi_num_transits  8422 non-null   float64
 14  koi_steff         9201 non-null   float64
 15  koi_slogg         9201 non-null   float64
 16  koi_smet          9178 non-null   float64


In [30]:
df_raw["koi_disposition"].value_counts(dropna=False)

koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2747
CANDIDATE         1978
Name: count, dtype: int64

In [31]:
df_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
kepid,9564.0,7.690628e+06,2.653459e+06,757450.000000,5.556034e+06,7.906892e+06,9.873066e+06,1.293514e+07
koi_period,9564.0,7.567136e+01,1.334744e+03,0.241843,2.733684e+00,9.752831e+00,4.071518e+01,1.299958e+05
koi_impact,9201.0,7.351055e-01,3.348832e+00,0.000000,1.970000e-01,5.370000e-01,8.890000e-01,1.008060e+02
koi_duration,9564.0,5.621606e+00,6.471554e+00,0.052000,2.437750e+00,3.792600e+00,6.276500e+00,1.385400e+02
koi_depth,9201.0,2.379134e+04,8.224268e+04,0.000000,1.599000e+02,4.211000e+02,1.473400e+03,1.541400e+06
koi_prad,9201.0,1.028918e+02,3.077639e+03,0.080000,1.400000e+00,2.390000e+00,1.493000e+01,2.003460e+05
koi_teq,9201.0,1.085386e+03,8.563512e+02,25.000000,5.390000e+02,8.780000e+02,1.379000e+03,1.466700e+04
koi_insol,9243.0,7.745737e+03,1.592047e+05,0.000000,2.015000e+01,1.416000e+02,8.702900e+02,1.094755e+07
koi_model_snr,9201.0,2.598950e+02,7.958066e+02,0.000000,1.200000e+01,2.300000e+01,7.800000e+01,9.054700e+03
koi_count,9564.0,1.406315e+00,8.732886e-01,1.000000,1.000000e+00,1.000000e+00,1.000000e+00,7.000000e+00


In [32]:
df_raw.isna().sum()

kepid                  0
kepoi_name             0
kepler_name         6817
koi_disposition        0
koi_period             0
koi_impact           363
koi_duration           0
koi_depth            363
koi_prad             363
koi_teq              363
koi_insol            321
koi_model_snr        363
koi_count              0
koi_num_transits    1142
koi_steff            363
koi_slogg            363
koi_smet             386
koi_srad             363
koi_smass            363
ra                     0
dec                    0
koi_kepmag             1
dtype: int64

In [33]:
# standardize column names and values
df = df_raw.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

# Strip whitespace from object/string columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

df.head()

,kepid,kepoi_name,kepler_name,koi_disposition,koi_period,koi_impact,koi_duration,koi_depth,koi_prad,koi_teq,...,koi_count,koi_num_transits,koi_steff,koi_slogg,koi_smet,koi_srad,koi_smass,ra,dec,koi_kepmag
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,9.488036,0.146,2.95750,615.8,2.26,793.0,...,2,142.0,5455.0,4.467,0.14,0.927,0.919,291.93423,48.141651,15.347
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,54.418383,0.586,4.50700,874.8,2.83,443.0,...,2,25.0,5455.0,4.467,0.14,0.927,0.919,291.93423,48.141651,15.347
2,10811496,K00753.01,NaN,CANDIDATE,19.899140,0.969,1.78220,10829.0,14.60,638.0,...,1,56.0,5853.0,4.544,-0.18,0.868,0.961,297.00482,48.134129,15.436
3,10848459,K00754.01,NaN,FALSE POSITIVE,1.736952,1.276,2.40641,8079.2,33.46,1395.0,...,1,621.0,5805.0,4.564,-0.52,0.791,0.836,285.53461,48.285210,15.597
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,2.525592,0.701,1.65450,603.3,2.75,1406.0,...,1,515.0,6031.0,4.438,0.07,1.046,1.095,288.75488,48.226200,15.509


In [34]:
# Define binary supervised learning target
# To make it binary, we will drop koi_disposition = 'FALSE POSITIVE' and 
# only keep 'CANDIDATE' and 'CONFIRMED' as positive examples.
df = df[df["koi_disposition"] != "FALSE POSITIVE"].copy()

df["planet"] = np.where(
    df["koi_disposition"].isin(["CONFIRMED"]),
    1,
    0
)

df["planet_label"] = df["planet"].map({
    1: "confirmed",
    0: "candidate"
})

print("Original shape:", df_raw.shape)
print("Model dataset shape:", df.shape)

df[["koi_disposition", "planet", "planet_label"]].head()

Original shape: (9564, 22)
Model dataset shape: (4725, 24)


,koi_disposition,planet,planet_label
0,CONFIRMED,1,confirmed
1,CONFIRMED,1,confirmed
2,CANDIDATE,0,candidate
4,CONFIRMED,1,confirmed
5,CONFIRMED,1,confirmed


In [35]:
df['planet_label'].value_counts()

planet_label
confirmed    2747
candidate    1978
Name: count, dtype: int64

In [36]:
# a bit of imbalance. ~58% of confirmed planets agains ~42% of candidates
df["planet_label"].value_counts(normalize=True).round(3)

planet_label
confirmed    0.581
candidate    0.419
Name: proportion, dtype: float64

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4725 entries, 0 to 9562
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   kepid             4725 non-null   int64  
 1   kepoi_name        4725 non-null   object 
 2   kepler_name       2745 non-null   object 
 3   koi_disposition   4725 non-null   object 
 4   koi_period        4725 non-null   float64
 5   koi_impact        4619 non-null   float64
 6   koi_duration      4725 non-null   float64
 7   koi_depth         4619 non-null   float64
 8   koi_prad          4619 non-null   float64
 9   koi_teq           4619 non-null   float64
 10  koi_insol         4622 non-null   float64
 11  koi_model_snr     4619 non-null   float64
 12  koi_count         4725 non-null   int64  
 13  koi_num_transits  4252 non-null   float64
 14  koi_steff         4619 non-null   float64
 15  koi_slogg         4619 non-null   float64
 16  koi_smet          4616 non-null   float64
 17  

In [38]:
ID_COLUMNS = ["kepid", "kepoi_name", "kepler_name"]

TARGET_COLUMNS = [
    "koi_disposition",
    "planet_like",
    "planet_like_label",
]

NUMERIC_COLUMNS = [
    "koi_period",
    "koi_impact",
    "koi_duration",
    "koi_depth",
    "koi_prad",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_count",
    "koi_num_transits",
    "koi_steff",
    "koi_slogg",
    "koi_smet",
    "koi_srad",
    "koi_smass",
    "ra",
    "dec",
    "koi_kepmag",
]

for col in NUMERIC_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4725 entries, 0 to 9562
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   kepid             4725 non-null   int64  
 1   kepoi_name        4725 non-null   object 
 2   kepler_name       2745 non-null   object 
 3   koi_disposition   4725 non-null   object 
 4   koi_period        4725 non-null   float64
 5   koi_impact        4619 non-null   float64
 6   koi_duration      4725 non-null   float64
 7   koi_depth         4619 non-null   float64
 8   koi_prad          4619 non-null   float64
 9   koi_teq           4619 non-null   float64
 10  koi_insol         4622 non-null   float64
 11  koi_model_snr     4619 non-null   float64
 12  koi_count         4725 non-null   int64  
 13  koi_num_transits  4252 non-null   float64
 14  koi_steff         4619 non-null   float64
 15  koi_slogg         4619 non-null   float64
 16  koi_smet          4616 non-null   float64
 17  

In [39]:
duplicate_koi_count = df["kepoi_name"].duplicated().sum()
duplicate_kepid_count = df["kepid"].duplicated().sum()

print("Duplicated KOI names:", duplicate_koi_count)
print("Duplicated Kepler IDs:", duplicate_kepid_count)

Duplicated KOI names: 0
Duplicated Kepler IDs: 1114


In [40]:
if duplicate_koi_count > 0:
    duplicated_rows = df[df["kepoi_name"].duplicated(keep=False)]
    display(duplicated_rows.sort_values("kepoi_name").head(20))

# Drop exact duplicate KOI rows if any exist.
df_model = df.drop_duplicates(subset=["kepoi_name"], keep="first")
print("Shape after dropping duplicate kepoi_name rows:", df.shape)

Shape after dropping duplicate kepoi_name rows: (4725, 24)


In [41]:
df.isna().sum()

kepid                  0
kepoi_name             0
kepler_name         1980
koi_disposition        0
koi_period             0
koi_impact           106
koi_duration           0
koi_depth            106
koi_prad             106
koi_teq              106
koi_insol            103
koi_model_snr        106
koi_count              0
koi_num_transits     473
koi_steff            106
koi_slogg            106
koi_smet             109
koi_srad             106
koi_smass            106
ra                     0
dec                    0
koi_kepmag             0
planet                 0
planet_label           0
dtype: int64

In [42]:
df_dropna = df.copy().dropna()

print(df_dropna.shape)

(2732, 24)


In [43]:
processed_file = PROCESSED_DIR / "kepler_koi_binary.csv"

df.to_csv(processed_file, index=False)

print("Saved processed dataset:", processed_file)
print("Final shape:", df.shape)

Saved processed dataset: d:\exoplanet-ml-api\data\processed\kepler_koi_binary.csv
Final shape: (4725, 24)


In [45]:
metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": "NASA Exoplanet Archive TAP service",
    "table": "cumulative",
    "query": " ".join(query.split()),
    "processed_file": str(processed_file.relative_to(PROJECT_ROOT)),
    "target_definition": {
        "planet = 1": ["CONFIRMED"],
        "planet = 0": ["CANDIDATE"],
        "removed": ["FALSE POSITIVE","NOT DISPOSITIONED"],
    },
    "id_columns": ID_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "numeric_columns": NUMERIC_COLUMNS,
    "notes": [
        "This is a simplified binary classification problem for an end-to-end ML engineering pilot.",
        "Obvious leakage columns such as false-positive flags, disposition scores, comments, and vetting labels are not used.",
        "RA and Dec are downloaded for EDA but excluded from the first feature set.",
    ],
}

metadata_file = REPORTS_DIR / "data_metadata.json"

with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("Saved metadata:", metadata_file)

Saved metadata: d:\exoplanet-ml-api\data\reports\data_metadata.json
